# Hausverwaltungs-Chatbot mit Langfuse-Tracing

Ein Telefon-Agent für eine Hausverwaltung, der in drei LLM-Schritten arbeitet —
und dabei jeden Schritt als **Trace** an Langfuse meldet, damit man im Dashboard
nachvollziehen kann, was das Modell wann entschieden hat.

1. **Begrüßung** (`step-greeting`) — freundlich antworten, ggf. nach dem Namen fragen
2. **Verifizierung** (`step-auth`) — Name + Adresse prüfen, strukturierte Antwort
3. **Routing** (`step-routing`) — Anliegen einer von fünf Abteilungen zuordnen

Alle drei Schritte eines Anrufs landen unter **einem gemeinsamen Trace**.

## Datenfluss

Wichtig für das Verständnis: Die LLM-Aufrufe sind **zustandslos** und bekommen die
Ausgaben der vorherigen LLMs NICHT als Kontext. Jeder Aufruf erhält nur die rohen
Nutzereingaben.

```
    opening (Anliegen des Mieters)
            |
            v
    +---------------------+  LLM-Input: opening
    | 1. step-greeting    |--------------------> greeting (str)
    +---------------------+                      nur ausgegeben, wird nicht
            |                                    weitergereicht
    name, address
            v
    +---------------------+  LLM-Input: "Name: ...\nAdresse: ..."
    | 2. step-auth        |--------------------> AuthResult
    +---------------------+                      genutzt: verified (Abbruch?),
            |                                    customer_name (Anrede)
      verified? --nein--> Ende (Tag "auth-failed")
            | ja
    issue (konkretes Anliegen)
            v
    +---------------------+  LLM-Input: transcript =
    | 3. step-routing     |  opening + name + address + issue
    +---------------------+--------------------> RoutingDecision
            |
            v
    4. Tags/Metadaten an den Trace haengen, Ausgabe an den Mieter
```

## Setup: Pakete importieren

Der Import `from langfuse.openai import openai` ist der zentrale Trick: Dieser
Wrapper verhält sich exakt wie das normale `openai`-Paket, fängt aber jeden
Aufruf ab und schickt Prompt, Antwort, Modell, Token-Zahl und Dauer automatisch
an Langfuse. Am eigentlichen API-Code ändert sich dadurch **nichts**.

In [ ]:
import os
import uuid

from dotenv import load_dotenv
from pydantic import BaseModel

# Der Langfuse-Wrapper faengt alle OpenAI-Aufrufe ab und traced sie automatisch
from langfuse.openai import openai
from langfuse import get_client, observe, propagate_attributes

## Konfiguration

Aus der `.env`-Datei kommen zwei Dinge: der OpenAI-Schlüssel (`OPENAI_API_KEY`)
und die Langfuse-Zugangsdaten (`LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`,
`LANGFUSE_HOST`). `get_client()` liest sie selbst aus der Umgebung — man muss
nichts übergeben.

`auth_check()` sagt sofort, ob die Verbindung zu Langfuse steht. Ohne diesen
Test merkt man einen Tippfehler im Schlüssel erst daran, dass das Dashboard
leer bleibt.

In [ ]:
load_dotenv()

model = os.getenv("LLM_MODEL", "gpt-4o-mini")
langfuse = get_client()

print(f"Modell: {model}")
print(f"Langfuse-Verbindung ok: {langfuse.auth_check()}")

## Die Abteilungen und die Datenmodelle

`DEPARTMENTS` ist die Wissensbasis des Routings — die Schlüssel landen später
wörtlich im System-Prompt, damit das Modell weiß, wohin es überhaupt verteilen darf.

Die beiden Pydantic-Klassen sind mehr als Typ-Deklarationen: Sie werden gleich als
`response_format` an die API übergeben. Das Modell muss dann JSON liefern, das
exakt zu diesen Feldern passt — kein Parsen von Freitext, keine
Überraschungen. Bei den Feldern lohnt der genaue Blick, denn sie steuern den
weiteren Programmablauf: `verified` entscheidet über den Abbruch, `department`
über das Ziel.

In [ ]:
DEPARTMENTS = {
    "rental-contracts":     "Mietverträge — Fragen zum Mietvertrag, Verlängerungen, Änderungen",
    "terminations-moveout": "Kündigungen & Auszug — Kündigungen, Auszugstermine, Kautionsrückzahlung",
    "tenant-complaints":    "Mieterbeschwerden — Lärm, Nachbarschaftsstreit, allgemeine Beschwerden",
    "energy-heating":       "Energie & Heizung — Heizungsausfälle, Warmwasser, Nebenkostenabrechnung",
    "repairs-maintenance":  "Reparaturen & Instandhaltung — defekte Einrichtungen, Gebäudeschäden, allgemeine Reparaturen",
}


class AuthResult(BaseModel):
    verified: bool
    customer_name: str
    reason: str


class RoutingDecision(BaseModel):
    department: str        # einer der fuenf Schluessel oben
    routing_reason: str    # warum diese Abteilung — wichtig fuer spaetere Analysen
    issue_summary: str
    confidence: str        # "niedrig" / "mittel" / "hoch"

## Schritt 1: Begrüßung

Ein ganz gewöhnlicher Chat-Aufruf: System-Prompt legt die Rolle fest, die
Mieter-Nachricht kommt als User-Message dazu, zurück kommt **freier Text** aus
`response.choices[0].message.content`.

Neu ist nur der Dekorator `@observe(name="step-greeting")`. Er macht aus dem
Funktionsaufruf einen **Span** im Trace: Langfuse misst die Dauer, merkt sich
Argumente und Rückgabewert und hängt den darin stattfindenden OpenAI-Aufruf als
Kind darunter. Im Dashboard sieht man später also die Verschachtelung
`step-greeting` → OpenAI-Call.

In [ ]:
@observe(name="step-greeting")       # Type-Span: Begruessung
def greet_and_collect_name(customer_message: str) -> str:
    response = openai.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": (
                "Du bist eine freundliche Empfangskraft bei der Hausverwaltung. "
                "Begrüße den Mieter herzlich und antworte immer auf Deutsch. "
                "Falls er seinen Namen noch nicht genannt hat, frage danach."
            )},
            {"role": "user", "content": customer_message}
        ]
    )
    return response.choices[0].message.content

In [ ]:
opening = "Guten Tag, bei mir in der Wohnung ist die Heizung ausgefallen."

greeting = greet_and_collect_name(opening)
print(greeting)

## Schritt 2: Verifizierung

Hier kommen zwei neue Zutaten dazu.

**Strukturierte Ausgabe:** Statt `chat.completions.create` wird
`beta.chat.completions.parse` mit `response_format=AuthResult` aufgerufen. Die
Antwort steckt dann nicht in `.content`, sondern fertig geparst in
`.parsed` — ein echtes `AuthResult`-Objekt, dessen `verified`-Feld man direkt in
einem `if` verwenden kann.

**Manueller Sub-Span:** Der Block `with langfuse.start_as_current_observation(...)`
erzeugt eine eigene Beobachtung *innerhalb* der Funktion. Das ist nützlich für
Logik, die gar kein LLM benutzt — hier eine simple Plausibilitätsprüfung der
Adresse. Über `span.update(metadata=...)` landen beliebige Werte im Dashboard,
nach denen man später filtern kann.

Hinweis: `is_plausible` wird bewusst nur protokolliert, nicht als Abbruchkriterium
verwendet — die Entscheidung trifft allein das Modell.

In [ ]:
@observe(name="step-auth")
def verify_tenant(name: str, address: str) -> AuthResult:
    # Manueller Sub-Span innerhalb der @observe-Funktion
    with langfuse.start_as_current_observation(name="address-format-check") as span:
        is_plausible = len(address.split()) >= 2
        span.update(metadata={"raw_address": address, "passed_format": is_plausible})

    response = openai.beta.chat.completions.parse(
        model=model,
        messages=[
            {"role": "system", "content": (
                "Du simulierst ein Mieter-Verifizierungssystem. "
                "Wenn die Adresse plausibel klingt (Straßenname + Hausnummer + Stadt), "
                "markiere den Mieter als verifiziert. "
                "Schreibe die Begründung (reason) auf Deutsch."
            )},
            {"role": "user", "content": f"Name: {name}\nAdresse: {address}"}
        ],
        response_format=AuthResult
    )
    return response.choices[0].message.parsed

In [ ]:
name = "Anna Schmidt"
address = "Hauptstraße 12, 10115 Berlin"

auth = verify_tenant(name, address)
print(auth)                 # ein AuthResult-Objekt, kein Text
print()
print("verified:     ", auth.verified)
print("customer_name:", auth.customer_name)
print("reason:       ", auth.reason)

In [ ]:
# Gegentest: eine unbrauchbare Adresse. Das Modell sollte verified=False liefern.
print(verify_tenant("Anna Schmidt", "keine Ahnung"))

## Schritt 3: Routing

Der System-Prompt wird hier **dynamisch** zusammengebaut: `dept_list` rendert das
`DEPARTMENTS`-Dictionary als Aufzählung in den Prompt hinein. Neue Abteilung
anlegen heißt damit nur, den Dictionary-Eintrag zu ergänzen — der Prompt zieht
automatisch nach.

Als User-Message geht das gesamte Transkript rein. Das ist die einzige Stelle,
an der die vorher gesammelten Informationen zusammenlaufen: `opening`, `name`,
`address` und `issue` als ein String. Zurück kommt wieder ein geparstes
Pydantic-Objekt.

In [ ]:
@observe(name="step-routing")
def route_to_department(transcript: str) -> RoutingDecision:
    dept_list = "\n".join(f"- {key}: {desc}" for key, desc in DEPARTMENTS.items())
    response = openai.beta.chat.completions.parse(
        model=model,
        messages=[
            {"role": "system", "content": (
                "Du leitest Mieteranfragen bei der Hausverwaltung an die richtige Abteilung weiter. "
                "Wähle genau einen Abteilungs-Schlüssel aus dieser Liste:\n" + dept_list + "\n"
                "Schreibe routing_reason und issue_summary auf Deutsch. "
                "confidence muss genau einer dieser Werte sein: \"niedrig\", \"mittel\", \"hoch\"."
            )},
            {"role": "user", "content": transcript}
        ],
        response_format=RoutingDecision
    )
    return response.choices[0].message.parsed

In [ ]:
issue = "Die Heizung ist seit gestern komplett kalt, auch das Warmwasser fehlt."

transcript = f"Eröffnung: {opening}\nName: {name}\nAdresse: {address}\nAnliegen: {issue}"
print(transcript)
print("\n" + "-" * 60 + "\n")

routing = route_to_department(transcript)
print("department:    ", routing.department)
print("routing_reason:", routing.routing_reason)
print("issue_summary: ", routing.issue_summary)
print("confidence:    ", routing.confidence)

## Alles zusammen: ein Trace pro Anruf

Bis hierhin hat jeder Aufruf seinen **eigenen** Trace erzeugt — praktisch zum
Ausprobieren, aber im Dashboard sieht man nicht, was zu welchem Anruf gehörte.
`handle_call` ist wieder mit `@observe` dekoriert und wird damit zum
**Eltern-Trace**, unter dem die drei Schritte als Kinder einsortiert werden.

`propagate_attributes` reicht Attribute an alles weiter, was innerhalb des
`with`-Blocks passiert:

- **`session_id`** klammert alle Schritte eines Anrufs zusammen. Bei einem
  echten Chatbot würde man hier die Konversations-ID des Nutzers einsetzen,
  dann lassen sich mehrere Anrufe derselben Person im Dashboard gruppieren.
- **`tags`** sind Stichworte zum Filtern — etwa alle abgebrochenen Anrufe
  (`auth-failed`) oder alle mit niedriger Konfidenz.
- **`metadata`** sind freie Schlüssel-Wert-Paare für spätere Auswertungen,
  z. B. "wie oft ging es zu welcher Abteilung?".

Der Aufruf am Ende ist der eigentliche Zweck des Ganzen: Erst dadurch wird das
Routing-Ergebnis **am Trace** sichtbar und nicht nur in der Konsole.

> Im Skript `traceability.py` holt `handle_call()` die vier Werte über `input()`
> vom Terminal. Hier stehen sie als Parameter, damit die Zelle ohne Warten auf
> Eingaben durchläuft und sich leicht wiederholt ausführen lässt.

In [ ]:
@observe(name="tenant-routing-call")
def handle_call(opening: str, name: str, address: str, issue: str):
    session_id = uuid.uuid4().hex[:8]
    with propagate_attributes(session_id=session_id):
        greeting = greet_and_collect_name(opening)
        print(f"Agent: {greeting}")

        auth = verify_tenant(name, address)
        if not auth.verified:
            with propagate_attributes(tags=["auth-failed"]):
                print("Agent: Es tut mir leid, ich konnte Ihre Angaben nicht verifizieren.")
            return None

        print(f"\nAgent: Vielen Dank, {auth.customer_name}. Wie kann ich Ihnen heute helfen?")
        print(f"Mieter: {issue}")

        transcript = f"Eröffnung: {opening}\nName: {name}\nAdresse: {address}\nAnliegen: {issue}"
        routing = route_to_department(transcript)

        # Routing-Ergebnis am uebergeordneten Trace anhaengen (fuers Filtern im Dashboard)
        with propagate_attributes(
            tags=[routing.department, f"confidence-{routing.confidence}"],
            metadata={
                "routing_department": routing.department,
                "routing_reason": routing.routing_reason,
                "confidence": routing.confidence,
                "customer_name": auth.customer_name,
            }
        ):
            print(f"\n✅ Weiterleitung an: {DEPARTMENTS[routing.department]}")
            print(f"   Grund: {routing.routing_reason}")
            print(f"   Konfidenz: {routing.confidence}")
        return routing

In [ ]:
routing = handle_call(
    opening="Guten Tag, bei mir in der Wohnung ist die Heizung ausgefallen.",
    name="Anna Schmidt",
    address="Hauptstraße 12, 10115 Berlin",
    issue="Die Heizung ist seit gestern komplett kalt, auch das Warmwasser fehlt.",
)

In [ ]:
# Der Abbruch-Pfad: unbrauchbare Adresse -> verified=False -> Tag "auth-failed"
handle_call(
    opening="Hallo, ich habe eine Frage zu meinem Mietvertrag.",
    name="Max Mustermann",
    address="xyz",
    issue="Ich möchte den Vertrag um zwei Jahre verlängern.",
)

## Traces absenden

Langfuse sammelt die Daten im Hintergrund und schickt sie gebündelt los. Ein
Skript erledigt das beim Beenden — ein Notebook-Kernel läuft aber weiter, deshalb
muss man hier selbst `flush()` aufrufen. Sonst wartet man vergeblich auf Traces
im Dashboard.

Faustregel: nach jedem Experiment ausführen, das man sich ansehen möchte.

In [ ]:
langfuse.flush()
print("Traces gesendet — jetzt im Langfuse-Dashboard sichtbar.")

## Experimente

- Ein Anliegen formulieren, das **zwischen zwei Abteilungen** liegt (z. B. "mein
  Nachbar heizt nicht und dadurch schimmelt meine Wand") — welche Abteilung wird
  gewählt, und sinkt die `confidence`?
- Eine Abteilung aus `DEPARTMENTS` entfernen und Schritt 3 neu ausführen:
  Der Prompt ändert sich automatisch mit.
- `confidence` ist bisher nur ein `str` — als
  `Literal["niedrig", "mittel", "hoch"]` deklarieren und beobachten, dass die
  API den Wert dann erzwingt statt ihn nur zu erbitten.
- Mehrere Anrufe mit derselben `session_id` durchführen (Parameter in
  `handle_call` durchreichen) und im Dashboard nach der Session filtern.
- Im Dashboard einen Trace öffnen und die Verschachtelung ansehen:
  `tenant-routing-call` → `step-auth` → `address-format-check`.